# nb_bronze_silver_gold_sec_edgar

SEC EDGAR → Bronze JSON → Silver facts → Gold balance sheet / income statement / cash flow for **NVIDIA (CIK 0001045810)**.

**Attach lakehouse:** `NVIDIALakehouse` (schema-enabled) in workspace `NVIDIA-BalanceSheet-AI` before Run all.

**Source:** `data.sec.gov` companyfacts + submissions (User-Agent required; ≤10 req/s).

In [ ]:
# Parameters
CIK = "0001045810"
CIK_INT = "1045810"
SEC_USER_AGENT = "DataPat.ai patricrc10@gmail.com"
COMPANYFACTS_URL = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{CIK}.json"
SUBMISSIONS_URL = f"https://data.sec.gov/submissions/CIK{CIK}.json"
FILES_ROOT = "/lakehouse/default/Files"
BRONZE_PREFIX = f"bronze/sec_edgar/cik={CIK}"

In [ ]:
# Imports + Spark write prefs (gold-friendly)
from __future__ import annotations

import json
import time
from datetime import date, datetime
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from pyspark.sql import functions as F

spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.databricks.delta.optimizeWrite.binSize", "1g")

HEADERS = {
    "User-Agent": SEC_USER_AGENT,
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov",
}
INGEST_DATE = date.today().isoformat()
print("ingest_date:", INGEST_DATE)
print("User-Agent:", SEC_USER_AGENT)

In [ ]:
# Curated concept map → gold.dim_line_item
# (statement, line_item, concept, expected_period_type)
CONCEPT_MAP = [
    ("balance_sheet", "Total Assets", "Assets", "instant"),
    ("balance_sheet", "Current Assets", "AssetsCurrent", "instant"),
    ("balance_sheet", "Cash & Equivalents", "CashAndCashEquivalentsAtCarryingValue", "instant"),
    ("balance_sheet", "Marketable Securities (Current)", "MarketableSecuritiesCurrent", "instant"),
    ("balance_sheet", "Accounts Receivable (Net, Current)", "AccountsReceivableNetCurrent", "instant"),
    ("balance_sheet", "Inventory (Net)", "InventoryNet", "instant"),
    ("balance_sheet", "Total Liabilities", "Liabilities", "instant"),
    ("balance_sheet", "Current Liabilities", "LiabilitiesCurrent", "instant"),
    ("balance_sheet", "Long-Term Debt (Noncurrent)", "LongTermDebtNoncurrent", "instant"),
    ("balance_sheet", "Operating Lease Liability (Current)", "OperatingLeaseLiabilityCurrent", "instant"),
    ("balance_sheet", "Operating Lease Liability (Noncurrent)", "OperatingLeaseLiabilityNoncurrent", "instant"),
    ("balance_sheet", "Stockholders' Equity", "StockholdersEquity", "instant"),
    ("balance_sheet", "Liabilities + Stockholders' Equity", "LiabilitiesAndStockholdersEquity", "instant"),
    ("income_statement", "Revenue (Contract, ex. tax)", "RevenueFromContractWithCustomerExcludingAssessedTax", "duration"),
    ("income_statement", "Revenue", "Revenues", "duration"),
    ("income_statement", "Cost of Revenue", "CostOfRevenue", "duration"),
    ("income_statement", "Cost of Goods & Services Sold", "CostOfGoodsAndServicesSold", "duration"),
    ("income_statement", "Gross Profit", "GrossProfit", "duration"),
    ("income_statement", "R&D Expense", "ResearchAndDevelopmentExpense", "duration"),
    ("income_statement", "Operating Income (Loss)", "OperatingIncomeLoss", "duration"),
    ("income_statement", "Net Income (Loss)", "NetIncomeLoss", "duration"),
    ("income_statement", "EPS Diluted", "EarningsPerShareDiluted", "duration"),
    ("cash_flow", "Cash from Operating Activities", "NetCashProvidedByUsedInOperatingActivities", "duration"),
    ("cash_flow", "Cash from Investing Activities", "NetCashProvidedByUsedInInvestingActivities", "duration"),
    ("cash_flow", "Cash from Financing Activities", "NetCashProvidedByUsedInFinancingActivities", "duration"),
    ("cash_flow", "CapEx (PPE)", "PaymentsToAcquirePropertyPlantAndEquipment", "duration"),
    ("cash_flow", "Share Repurchases", "PaymentsForRepurchaseOfCommonStock", "duration"),
    ("cash_flow", "Dividends Paid", "PaymentsOfDividends", "duration"),
    ("cash_flow", "Dividends Paid (Common)", "PaymentsOfDividendsCommonStock", "duration"),
]
TAXONOMIES = ("us-gaap", "dei")

## 1 · Bronze — land raw SEC JSON once

Fabric Spark cannot read arbitrary HTTPS URLs. Download with `requests` (declared User-Agent), write under `Files/bronze/sec_edgar/...`, then parse from the lakehouse path. Idempotent: skip overwrite if both files already exist for today's `ingest_date`.

In [ ]:
def fetch_json(url: str) -> dict:
    r = requests.get(url, headers=HEADERS, timeout=120)
    r.raise_for_status()
    time.sleep(0.2)  # well under SEC fair-access 10 req/s
    return r.json()


bronze_dir = Path(FILES_ROOT) / BRONZE_PREFIX / f"ingest_date={INGEST_DATE}"
bronze_dir.mkdir(parents=True, exist_ok=True)
cf_path = bronze_dir / "companyfacts.json"
sub_path = bronze_dir / "submissions.json"

if cf_path.exists() and sub_path.exists():
    print(f"Bronze already landed for {INGEST_DATE}; reusing cache")
    companyfacts = json.loads(cf_path.read_text())
    submissions = json.loads(sub_path.read_text())
else:
    print("Fetching companyfacts + submissions from data.sec.gov …")
    companyfacts = fetch_json(COMPANYFACTS_URL)
    submissions = fetch_json(SUBMISSIONS_URL)
    cf_path.write_text(json.dumps(companyfacts))
    sub_path.write_text(json.dumps(submissions))
    print(f"wrote {cf_path} ({cf_path.stat().st_size:,} bytes)")
    print(f"wrote {sub_path} ({sub_path.stat().st_size:,} bytes)")

print("entity:", companyfacts.get("entityName"), "| cik:", companyfacts.get("cik"))
print("us-gaap concepts:", len(companyfacts.get("facts", {}).get("us-gaap", {})))

## 2 · Silver — `silver.financial_fact`

One row per taxonomy × concept × unit × observation (start/end/val/fy/fp/form/accn/filed/frame). Adds `period_type` (`instant`|`duration`) and `duration_days`.

In [ ]:
def flatten_companyfacts(companyfacts: dict) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []
    facts = companyfacts.get("facts", {})
    for taxonomy in TAXONOMIES:
        for concept, payload in facts.get(taxonomy, {}).items():
            label = payload.get("label")
            for unit, observations in payload.get("units", {}).items():
                for obs in observations:
                    start = obs.get("start")
                    end = obs.get("end")
                    period_type = "instant" if start is None else "duration"
                    duration_days = None
                    if start and end:
                        try:
                            duration_days = (
                                datetime.strptime(end, "%Y-%m-%d").date()
                                - datetime.strptime(start, "%Y-%m-%d").date()
                            ).days
                        except ValueError:
                            duration_days = None
                    rows.append(
                        {
                            "cik": CIK,
                            "entity_name": companyfacts.get("entityName"),
                            "taxonomy": taxonomy,
                            "concept": concept,
                            "label": label,
                            "unit": unit,
                            "start": start,
                            "end": end,
                            "val": obs.get("val"),
                            "fy": obs.get("fy"),
                            "fp": obs.get("fp"),
                            "form": obs.get("form"),
                            "accn": obs.get("accn"),
                            "filed": obs.get("filed"),
                            "frame": obs.get("frame"),
                            "period_type": period_type,
                            "duration_days": duration_days,
                            "ingest_date": INGEST_DATE,
                        }
                    )
    return pd.DataFrame(rows)


pdf_facts = flatten_companyfacts(companyfacts)
print("silver.financial_fact rows:", len(pdf_facts))
print(pdf_facts["taxonomy"].value_counts().to_dict())
print(pdf_facts["period_type"].value_counts().to_dict())

## 3 · Dimensions + Gold statements

- `gold.dim_filing` from submissions (accn, form, filingDate, reportDate, primaryDocument, sec_url)
- `gold.dim_line_item` curated map
- `gold.balance_sheet` / `gold.income_statement` / `gold.cash_flow` (long format)

Rules: USD (+ USD/shares for EPS); 10-K / 10-Q (+ amendments); BS = instant, IS/CF = duration; restatements keep latest `filed` per concept+period window; `is_period_primary` when `end == filing reportDate` (drops prior-year comparatives from “current”); 10-Q YTD flagged when `duration_days > 100` (raw YTD kept — no invented quarters).

In [ ]:
def build_dim_filing(submissions: dict) -> pd.DataFrame:
    recent = submissions.get("filings", {}).get("recent", {})
    n = len(recent.get("accessionNumber", []))
    rows = []
    for i in range(n):
        accn = recent["accessionNumber"][i]
        primary = recent.get("primaryDocument", [None] * n)[i]
        accn_nodash = accn.replace("-", "")
        rows.append(
            {
                "accn": accn,
                "form": recent["form"][i],
                "filingDate": recent["filingDate"][i],
                "reportDate": recent.get("reportDate", [None] * n)[i],
                "primaryDocument": primary,
                "sec_url": (
                    f"https://www.sec.gov/Archives/edgar/data/{CIK_INT}/{accn_nodash}/{primary}"
                    if primary
                    else None
                ),
            }
        )
    return pd.DataFrame(rows)


def build_dim_line_item() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {
                "statement": s,
                "line_item": li,
                "concept": c,
                "expected_period_type": pt,
            }
            for s, li, c, pt in CONCEPT_MAP
        ]
    )


def build_statement(facts: pd.DataFrame, statement: str, filings: pd.DataFrame) -> pd.DataFrame:
    concepts = [c for s, li, c, pt in CONCEPT_MAP if s == statement]
    line_map = {c: li for s, li, c, pt in CONCEPT_MAP if s == statement}
    pt_map = {c: pt for s, li, c, pt in CONCEPT_MAP if s == statement}

    usd = facts[
        (facts["concept"].isin(concepts))
        & (facts["unit"].isin(["USD", "USD/shares"]))
        & (facts["form"].isin(["10-K", "10-Q", "10-K/A", "10-Q/A"]))
    ].copy()
    if usd.empty:
        return usd

    usd = usd[usd["period_type"] == usd["concept"].map(pt_map)]
    usd["form_base"] = usd["form"].str.replace("/A", "", regex=False)

    key_cols = ["concept", "start", "end", "fy", "fp", "form_base", "unit"]
    usd = usd.sort_values(["filed", "accn"]).drop_duplicates(subset=key_cols, keep="last")

    usd = usd.merge(
        filings[["accn", "reportDate"]].rename(columns={"reportDate": "filing_report_date"}),
        on="accn",
        how="left",
    )
    usd["is_period_primary"] = usd["filing_report_date"].notna() & (
        usd["end"] == usd["filing_report_date"]
    )
    if (~usd["is_period_primary"]).any():
        has_primary = (
            usd[usd["is_period_primary"]]
            .groupby(["concept", "fy", "fp", "accn"], dropna=False)
            .size()
        )
        fallback_idx = (
            usd[~usd["is_period_primary"]]
            .sort_values("end")
            .groupby(["concept", "fy", "fp", "accn"], dropna=False)
            .tail(1)
            .index
        )
        for idx in fallback_idx:
            r = usd.loc[idx]
            key = (r["concept"], r["fy"], r["fp"], r["accn"])
            if key not in getattr(has_primary, "index", []):
                usd.at[idx, "is_period_primary"] = True

    usd["is_ytd"] = False
    if statement in ("cash_flow", "income_statement"):
        mask = (
            (usd["form_base"] == "10-Q")
            & usd["duration_days"].notna()
            & (usd["duration_days"] > 100)
        )
        usd.loc[mask, "is_ytd"] = True

    usd["is_latest_filing"] = False
    primary = usd[usd["is_period_primary"]]
    if not primary.empty:
        latest_idx = (
            primary.sort_values(["filed", "accn"])
            .groupby(["concept", "fy", "fp"], dropna=False)
            .tail(1)
            .index
        )
        usd.loc[latest_idx, "is_latest_filing"] = True

    return pd.DataFrame(
        {
            "statement": statement,
            "line_item": usd["concept"].map(line_map),
            "concept": usd["concept"],
            "fiscal_year": usd["fy"].astype("Int64"),
            "fiscal_period": usd["fp"],
            "period_end": usd["end"],
            "period_start": usd["start"],
            "value_usd": usd["val"],
            "unit": usd["unit"],
            "form": usd["form"],
            "accn": usd["accn"],
            "filed": usd["filed"],
            "is_latest_filing": usd["is_latest_filing"],
            "is_period_primary": usd["is_period_primary"],
            "is_ytd": usd["is_ytd"],
            "duration_days": usd["duration_days"],
            "period_type": usd["period_type"],
            "ingest_date": INGEST_DATE,
        }
    ).reset_index(drop=True)


pdf_filings = build_dim_filing(submissions)
pdf_line_items = build_dim_line_item()
pdf_bs = build_statement(pdf_facts, "balance_sheet", pdf_filings)
pdf_inc = build_statement(pdf_facts, "income_statement", pdf_filings)
pdf_cf = build_statement(pdf_facts, "cash_flow", pdf_filings)

present = set(pdf_facts["concept"].unique())
missing = [(s, li, c) for s, li, c, pt in CONCEPT_MAP if c not in present]
print("missing curated concepts:", missing or "none")
print("rows BS/IS/CF:", len(pdf_bs), len(pdf_inc), len(pdf_cf))

## 4 · Write Delta tables (schema-enabled lakehouse)

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")


def overwrite_table(pdf: pd.DataFrame, fqn: str) -> int:
    if pdf is None or pdf.empty:
        print(f"{fqn}: empty — skip")
        return 0
    sdf = spark.createDataFrame(pdf)
    (
        sdf.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(fqn)
    )
    n = sdf.count()
    print(f"{fqn}: {n:,} rows")
    return n


n_facts = overwrite_table(pdf_facts, "silver.financial_fact")
n_filings = overwrite_table(pdf_filings, "gold.dim_filing")
n_lines = overwrite_table(pdf_line_items, "gold.dim_line_item")
n_bs = overwrite_table(pdf_bs, "gold.balance_sheet")
n_inc = overwrite_table(pdf_inc, "gold.income_statement")
n_cf = overwrite_table(pdf_cf, "gold.cash_flow")

## 5 · Quality checks

In [ ]:
# Identity: Assets == LiabilitiesAndStockholdersEquity on latest primary periods
latest_bs = pdf_bs[pdf_bs["is_latest_filing"]].copy()
assets = latest_bs[latest_bs["concept"] == "Assets"][
    ["fiscal_year", "fiscal_period", "period_end", "value_usd", "accn", "form"]
].rename(columns={"value_usd": "assets", "accn": "assets_accn"})
liabeq = latest_bs[latest_bs["concept"] == "LiabilitiesAndStockholdersEquity"][
    ["fiscal_year", "fiscal_period", "period_end", "value_usd", "accn"]
].rename(columns={"value_usd": "liab_eq", "accn": "liab_eq_accn"})
idchk = assets.merge(liabeq, on=["fiscal_year", "fiscal_period", "period_end"], how="inner")
idchk["diff"] = idchk["assets"] - idchk["liab_eq"]
idchk["ok"] = idchk["diff"].abs() <= 1.0
print(f"Identity check: {int(idchk['ok'].sum())}/{len(idchk)} periods pass (tol=$1)")
display(idchk.sort_values("period_end").tail(12))

print("\nRow counts:")
for name, n in [
    ("silver.financial_fact", n_facts),
    ("gold.dim_filing", n_filings),
    ("gold.dim_line_item", n_lines),
    ("gold.balance_sheet", n_bs),
    ("gold.income_statement", n_inc),
    ("gold.cash_flow", n_cf),
]:
    print(f"  {name}: {n:,}")

print("\nLatest period per statement (primary + latest filing):")
for label, pdf in [
    ("balance_sheet", pdf_bs),
    ("income_statement", pdf_inc),
    ("cash_flow", pdf_cf),
]:
    sub = pdf[pdf["is_latest_filing"]]
    if sub.empty:
        print(f"  {label}: none")
        continue
    row = sub.sort_values(["period_end", "fiscal_year"]).iloc[-1]
    print(
        f"  {label}: FY{int(row['fiscal_year'])} {row['fiscal_period']} "
        f"end={row['period_end']} form={row['form']} accn={row['accn']}"
    )

# Sample FY key lines for the newest annual primary BS year
bs_fy = pdf_bs[(pdf_bs["fiscal_period"] == "FY") & (pdf_bs["is_latest_filing"])]
if not bs_fy.empty:
    latest_fy = int(bs_fy["fiscal_year"].max())
    print(f"\nFY{latest_fy} key lines (is_latest_filing):")
    for concept in [
        "Assets",
        "Liabilities",
        "StockholdersEquity",
        "CashAndCashEquivalentsAtCarryingValue",
        "Revenues",
        "NetIncomeLoss",
        "NetCashProvidedByUsedInOperatingActivities",
    ]:
        for pdf in (pdf_bs, pdf_inc, pdf_cf):
            hit = pdf[
                (pdf["concept"] == concept)
                & (pdf["fiscal_year"] == latest_fy)
                & (pdf["fiscal_period"] == "FY")
                & (pdf["is_latest_filing"])
            ]
            if not hit.empty:
                r = hit.iloc[0]
                print(
                    f"  {r['line_item']}: {float(r['value_usd']):,.0f} | "
                    f"end {r['period_end']} | {r['form']} {r['accn']}"
                )
                break

if missing:
    print("\nCurated concepts absent from companyfacts:")
    for s, li, c in missing:
        print(f"  [{s}] {c} ({li})")